In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

ImportError: cannot import name 'display' from 'IPython.core.display' (c:\Users\yazan\anaconda3\envs\tf_env\Lib\site-packages\IPython\core\display.py)

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [2]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv(r"C:\Users\yazan\my_ironhack\lab-natural-language-processing\data\kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [3]:
# Your code
from sklearn.model_selection import train_test_split 
x = data['text']
y = data['label']
X_train, X_test, y_train,  y_test = train_test_split(x,y)

## Data Preprocessing

In [4]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [5]:
# Your code
import re

def clean_html(text):
    
    # 1. remove JavaScript / CSS
    text = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)
    text = re.sub(r'<style.*?>.*?</style>', '', text, flags=re.DOTALL)

    # 2. remove HTML comments (before tags)
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)

    # 3. remove remaining HTML tags
    text = re.sub(r'<.*?>', '', text)

    return text

X_tr_clean1 = X_train.apply(clean_html)
X_te_clean1 = X_test.apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [6]:
# Your code


def clean_text(text):
    
    # 1. Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

    # 2. Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # 3. Remove single characters
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)

    # 4. Remove single characters from start
    text = re.sub(r'^\s*[a-zA-Z]\s+', ' ', text)

    # 5. Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)

    # 6. Remove prefixed 'b'
    text = re.sub(r'^b\s+', '', text)

    # 7. Convert to lowercase
    text = text.lower()

    return text.strip()

X_tr_clean2 = X_tr_clean1.apply(clean_text)
X_te_clean2 = X_te_clean1.apply(clean_text)


## Now let's work on removing stopwords
Remove the stopwords.

In [7]:
# Your code
from nltk import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt_tab')
nltk.download('stopwords')

stop_words = set(stopwords.words("english"))

def clean_sw(text):
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.lower() not in stop_words]
    return tokens

X_tr_clean3 = X_tr_clean2.apply(clean_sw)
X_te_clean3 = X_te_clean2.apply(clean_sw)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [8]:
# Your code
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()
def lemmatizing(text):
    
    text = [lemmatizer.lemmatize(word) for word in text]
    return text

X_tr_clean4 = X_tr_clean3.apply(lemmatizing)
X_te_clean4 = X_te_clean3.apply(lemmatizing)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yazan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd


spam_text = X_tr_clean4[data['label'] == 1].apply(lambda x: " ".join(x))
ham_text  = X_tr_clean4[data['label'] == 0].apply(lambda x: " ".join(x))


def get_top_words(text_data, n=10):

    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(text_data)

    words = vectorizer.get_feature_names_out()
    counts = X.sum(axis=0).A1

    df = pd.DataFrame({
        'word': words,
        'count': counts
    })

    return df.sort_values(by='count', ascending=False).head(n)


top10_spam = get_top_words(spam_text, 10)
top10_ham = get_top_words(ham_text, 10)

print(top10_spam)
print(top10_ham)

              word  count
12335        money    700
206        account    672
2141          bank    583
7006          fund    572
18915  transaction    395
2814      business    383
12443           mr    365
3934       country    365
12164      million    333
3652       company    300
           word  count
4937      state    120
5911      would     91
3921         pm     86
5343       time     84
3361         mr     78
4040  president     75
814        call     68
3642        one     68
3551      obama     63
5896       work     63


## Extra features

In [11]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

NameError: name 'data_train' is not defined

## How would you create a Bag of Words with the CountVectorizer method?

In [10]:
# Your code
X_train_text = X_tr_clean4.apply(lambda x: " ".join(x))
X_test_text  = X_te_clean4.apply(lambda x: " ".join(x))

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train_text)
X_test_vec  = vectorizer.transform(X_test_text)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

from sklearn.metrics import accuracy_score

print(accuracy_score(y_test, y_pred))

0.96


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [12]:
# Your code
all_clean = data['text'].apply(clean_html).apply(clean_text)
all_clean = all_clean.apply(clean_sw).apply(lemmatizing)
all_text = all_clean.apply(lambda x: " ".join(x))
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_tfidf = vectorizer.fit_transform(all_text)

print(X_tfidf.shape)

(1000, 29929)


### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code